In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv(r"C:\Users\victo\Downloads\spy_2020_2022.csv", low_memory=False)

In [ ]:
df.columns = df.columns.str.strip().str.strip('[]')
df = df.drop(['QUOTE_UNIXTIME', 'QUOTE_READTIME', 'QUOTE_TIME_HOURS', 'EXPIRE_UNIX', 'C_DELTA', 'C_GAMMA', 'C_VEGA', 'C_THETA', 'C_RHO', 'P_DELTA', 'P_GAMMA', 'P_VEGA', 'P_THETA', 'P_RHO'], axis=1)
df = df.applymap(lambda x: x.lstrip() if isinstance(x, str) else x)
df['EXPIRE_DATE'] = pd.to_datetime(df['EXPIRE_DATE'], errors='coerce')
df['QUOTE_DATE'] = pd.to_datetime(df['QUOTE_DATE'], errors='coerce')
date_cols = ['EXPIRE_DATE', 'QUOTE_DATE']
cols_to_exclude = ['EXPIRE_DATE', 'QUOTE_DATE', 'C_SIZE', 'P_SIZE']
num_cols = df.columns.difference(cols_to_exclude)
df[date_cols] = df[date_cols].apply(pd.to_datetime, format='%Y-%m-%d', errors='coerce')
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
df.head(n=10)

C:\Users\victo\AppData\Local\Temp\ipykernel_9412\1280743297.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lstrip() if isinstance(x, str) else x)


,QUOTE_DATE,UNDERLYING_LAST,EXPIRE_DATE,DTE,C_IV,C_VOLUME,C_LAST,C_SIZE,C_BID,C_ASK,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,STRIKE_DISTANCE,STRIKE_DISTANCE_PCT
0,2021-09-01,451.85,2021-09-01,0.0,NaN,1.0,182.65,1 x 1,181.09,182.31,270.0,0.0,0.01,0 x 2239,0.01,3.41249,3.0,181.9,0.402
1,2021-09-01,451.85,2021-09-01,0.0,NaN,NaN,0.00,1 x 1,176.09,177.31,275.0,0.0,0.01,0 x 2679,0.01,3.29737,0.0,176.9,0.391
2,2021-09-01,451.85,2021-09-01,0.0,NaN,25.0,157.75,1 x 1,171.09,172.31,280.0,0.0,0.01,0 x 2679,0.01,3.18330,11.0,171.9,0.380
3,2021-09-01,451.85,2021-09-01,0.0,NaN,25.0,153.45,1 x 1,166.14,167.30,285.0,0.0,0.01,0 x 2679,0.01,3.07217,50.0,166.9,0.369
4,2021-09-01,451.85,2021-09-01,0.0,NaN,25.0,147.76,1 x 1,161.09,162.31,290.0,0.0,0.01,0 x 2679,0.01,2.96230,0.0,161.9,0.358
5,2021-09-01,451.85,2021-09-01,0.0,NaN,25.0,143.46,200 x 200,156.10,157.30,295.0,0.0,0.01,0 x 2679,0.04,2.85433,1.0,156.9,0.347
6,2021-09-01,451.85,2021-09-01,0.0,NaN,24.0,138.51,200 x 200,151.10,152.30,300.0,0.0,0.01,0 x 2679,0.01,2.74842,3.0,151.9,0.336
7,2021-09-01,451.85,2021-09-01,0.0,NaN,24.0,134.11,1 x 2,146.15,147.31,305.0,0.0,0.01,0 x 2679,0.02,2.64392,0.0,146.9,0.325
8,2021-09-01,451.85,2021-09-01,0.0,NaN,NaN,0.00,200 x 200,141.10,142.30,310.0,0.0,0.01,0 x 2174,0.08,2.54076,1.0,141.9,0.314
9,2021-09-01,451.85,2021-09-01,0.0,NaN,10.0,137.09,200 x 200,136.10,137.30,315.0,0.0,0.01,0 x 2679,0.00,2.43886,NaN,136.9,0.303


In [ ]:
df = df.drop(['C_IV', 'C_VOLUME', 'C_LAST', 'C_SIZE', 'C_BID', 'C_ASK', 'STRIKE_DISTANCE', 'STRIKE_DISTANCE_PCT', 'EXPIRE_DATE'], axis=1)
df['moneyness'] = df['UNDERLYING_LAST'] / df['STRIKE']
df = df[df['P_BID'] > 0]  # filter out 0 bid options
df['midprice'] = (df['P_BID'] + df['P_ASK']) / 2.0
df = df[df['DTE'] > 0]  # filter out 0DTE options
df = df[df['P_VOLUME'] > 0] # filter out 0 volume options
df = df[df['P_IV'] < 3] # filter out extreme IV values
df = df[df['P_IV'] > 0] # filter out extreme IV values
df['T'] = df['DTE'] / 365.0
df.head(n=10)

,QUOTE_DATE,UNDERLYING_LAST,DTE,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,moneyness,midprice,T
157,2021-09-01,451.85,2.0,397.0,0.01,0.02,2040 x 4188,0.01,0.53525,452.0,1.138161,0.015,0.005479
158,2021-09-01,451.85,2.0,398.0,0.01,0.02,2359 x 4347,0.01,0.52504,11.0,1.135302,0.015,0.005479
160,2021-09-01,451.85,2.0,400.0,0.01,0.02,2326 x 3288,0.01,0.50651,20.0,1.129625,0.015,0.005479
162,2021-09-01,451.85,2.0,402.0,0.01,0.02,2286 x 2453,0.02,0.48794,2.0,1.124005,0.015,0.005479
164,2021-09-01,451.85,2.0,404.0,0.01,0.02,2246 x 2413,0.02,0.47024,3.0,1.118441,0.015,0.005479
165,2021-09-01,451.85,2.0,405.0,0.01,0.02,2226 x 3926,0.01,0.46041,55.0,1.115679,0.015,0.005479
167,2021-09-01,451.85,2.0,407.0,0.01,0.02,2186 x 3223,0.02,0.43999,1.0,1.110197,0.015,0.005479
168,2021-09-01,451.85,2.0,408.0,0.01,0.02,2171 x 3059,0.02,0.43037,2.0,1.107475,0.015,0.005479
169,2021-09-01,451.85,2.0,409.0,0.01,0.02,2162 x 3437,0.02,0.42194,1.0,1.104768,0.015,0.005479
170,2021-09-01,451.85,2.0,410.0,0.01,0.02,3092 x 3486,0.02,0.41266,13.0,1.102073,0.015,0.005479


In [ ]:
df['log_moneyness'] = np.log(df['moneyness'])
df['q'] = 0.015  # constant dividend yield
df.head(n=10)

,QUOTE_DATE,UNDERLYING_LAST,DTE,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,moneyness,midprice,T,log_moneyness,q
157,2021-09-01,451.85,2.0,397.0,0.01,0.02,2040 x 4188,0.01,0.53525,452.0,1.138161,0.015,0.005479,0.129414,0.015
158,2021-09-01,451.85,2.0,398.0,0.01,0.02,2359 x 4347,0.01,0.52504,11.0,1.135302,0.015,0.005479,0.126898,0.015
160,2021-09-01,451.85,2.0,400.0,0.01,0.02,2326 x 3288,0.01,0.50651,20.0,1.129625,0.015,0.005479,0.121886,0.015
162,2021-09-01,451.85,2.0,402.0,0.01,0.02,2286 x 2453,0.02,0.48794,2.0,1.124005,0.015,0.005479,0.116898,0.015
164,2021-09-01,451.85,2.0,404.0,0.01,0.02,2246 x 2413,0.02,0.47024,3.0,1.118441,0.015,0.005479,0.111935,0.015
165,2021-09-01,451.85,2.0,405.0,0.01,0.02,2226 x 3926,0.01,0.46041,55.0,1.115679,0.015,0.005479,0.109463,0.015
167,2021-09-01,451.85,2.0,407.0,0.01,0.02,2186 x 3223,0.02,0.43999,1.0,1.110197,0.015,0.005479,0.104537,0.015
168,2021-09-01,451.85,2.0,408.0,0.01,0.02,2171 x 3059,0.02,0.43037,2.0,1.107475,0.015,0.005479,0.102083,0.015
169,2021-09-01,451.85,2.0,409.0,0.01,0.02,2162 x 3437,0.02,0.42194,1.0,1.104768,0.015,0.005479,0.099635,0.015
170,2021-09-01,451.85,2.0,410.0,0.01,0.02,3092 x 3486,0.02,0.41266,13.0,1.102073,0.015,0.005479,0.097193,0.015


In [ ]:
rf = pd.read_csv(r"C:\Users\victo\Downloads\DTB3.csv", low_memory=False)
rf['observation_date'] = pd.to_datetime(rf['observation_date'], format='%Y-%m-%d', errors='coerce')
rf.rename(columns={'observation_date': 'date', 'DTB3': 'risk_free_rate'}, inplace=True)
rf['risk_free_rate'] /= 100
rf.sort_values('date', inplace=True)
rf.fillna(method='ffill', inplace=True)
rf.head(n=10)

C:\Users\victo\AppData\Local\Temp\ipykernel_9412\2741189155.py:6: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  rf.fillna(method='ffill', inplace=True)


,date,risk_free_rate
0,2015-05-08,0.0001
1,2015-05-11,0.0002
2,2015-05-12,0.0003
3,2015-05-13,0.0002
4,2015-05-14,0.0001
5,2015-05-15,0.0002
6,2015-05-18,0.0002
7,2015-05-19,0.0002
8,2015-05-20,0.0002
9,2015-05-21,0.0002


In [ ]:
# Merge on the 'date' column
df.rename(columns={'QUOTE_DATE': 'date'}, inplace=True)
df = df.merge(rf, on='date', how='left')

In [ ]:
# Sort by date
df = df.sort_values('date')

# Determine cutoff date (e.g., 80% point)
cutoff_date = df['date'].quantile(0.8)

# Create splits
train_df = df[df['date'] <= cutoff_date]
test_df = df[df['date'] > cutoff_date]

In [ ]:
len(train_df), len(test_df)

(1595555, 397454)

In [ ]:
from torch.utils.data import Dataset
from sklearn.preprocessing import StandardScaler

class OptionpricingDataset(Dataset):
    def __init__(self, df, scaler=None, fit_scaler=False):
        df = df.copy()
        self.scaler = scaler
        self.features_cols = ['log_moneyness', 'T', 'risk_free_rate', 'q', 'P_IV']
        self.target_col = ['midprice']
        X = df[self.features_cols].values.astype(np.float32)
        y = df[self.target_col].values.astype(np.float32).reshape(-1, 1)

        # apply scaling
        if scaler is None:
            self.scaler = StandardScaler()
            self.X = self.scaler.fit_transform(X)
        else:
            self.scaler = scaler
            if fit_scaler:
                self.X = self.scaler.fit_transform(X)
            else:
                self.X = self.scaler.transform(X)

        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
train_dataset = OptionpricingDataset(train_df, fit_scaler=True)
scaler = train_dataset.scaler  # Save the fitted scaler

test_dataset = OptionpricingDataset(test_df, scaler=scaler, fit_scaler=False)

In [ ]:
import torch.nn as nn

class OptionPricingNN(nn.Module):
    def __init__(self, input_dim, hidden_sizes=[64, 64, 64,]):
        super(OptionPricingNN, self).__init__()
        layers = []
        in_dim = input_dim

        for hidden_dim in hidden_sizes:
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            in_dim = hidden_dim

        layers.append(nn.Linear(in_dim, 1))  # Output layer
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


In [ ]:
import torch
from torch.utils.data import DataLoader

def train_model(model, train_dataset, test_dataset, epochs=200, batch_size=1024, lr=1e-3):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    def mape_loss(y_pred, y_true, eps=1e-6):
        return torch.mean(torch.abs((y_true - y_pred) / (y_true + eps)))
    criterion = mape_loss

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X_batch.size(0)

        avg_train_loss = train_loss / len(train_dataset)

        # Evaluation
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                test_loss += loss.item() * X_batch.size(0)

        avg_test_loss = test_loss / len(test_dataset)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.6f} | Test Loss: {avg_test_loss:.6f}")

    return model


In [ ]:
# Initialize model
input_dim = len(train_dataset.features_cols)
model = OptionPricingNN(input_dim=input_dim)

# Train model
trained_model = train_model(model, train_dataset, test_dataset, epochs=20)


Epoch 1/20 | Train Loss: 0.245332 | Test Loss: 1.077097
Epoch 2/20 | Train Loss: 0.101154 | Test Loss: 0.862576
Epoch 3/20 | Train Loss: 0.089868 | Test Loss: 1.563753
Epoch 4/20 | Train Loss: 0.083797 | Test Loss: 2.396622
Epoch 5/20 | Train Loss: 0.080287 | Test Loss: 3.663688
Epoch 6/20 | Train Loss: 0.077670 | Test Loss: 2.131800
Epoch 7/20 | Train Loss: 0.075136 | Test Loss: 2.854616
Epoch 8/20 | Train Loss: 0.072931 | Test Loss: 1.828747
Epoch 9/20 | Train Loss: 0.070787 | Test Loss: 2.539001
Epoch 10/20 | Train Loss: 0.069563 | Test Loss: 2.711907
Epoch 11/20 | Train Loss: 0.067014 | Test Loss: 1.765935
Epoch 12/20 | Train Loss: 0.066073 | Test Loss: 2.185887
Epoch 13/20 | Train Loss: 0.065057 | Test Loss: 1.833994
Epoch 14/20 | Train Loss: 0.063641 | Test Loss: 1.397804
Epoch 15/20 | Train Loss: 0.063015 | Test Loss: 1.786321
Epoch 16/20 | Train Loss: 0.061938 | Test Loss: 1.295840
Epoch 17/20 | Train Loss: 0.061454 | Test Loss: 0.915211
Epoch 18/20 | Train Loss: 0.061305 | Tes

In [ ]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def get_prediction_table(model, test_dataset, original_df=None, num_rows=10000000):
    model.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    X = test_dataset.X.to(device)
    y_true = test_dataset.y.cpu().numpy()
    
    with torch.no_grad():
        y_pred = model(X).cpu().numpy()

    # Compute errors
    abs_error = np.abs(y_true.flatten() - y_pred.flatten())
    rel_error = abs_error / np.maximum(y_true.flatten(), 1e-6)

    mae = np.mean(abs_error)
    rmse = np.sqrt(np.mean((y_true.flatten() - y_pred.flatten()) ** 2))
    rel_error_sum = np.sum(rel_error)

    print(f"\nMAE: {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"Sum of Relative Errors: {rel_error_sum:.6f}")

    # Create table
    data = {
        'Actual Price': y_true.flatten(),
        'Predicted Price': y_pred.flatten(),
        'Absolute Error': abs_error,
        'Relative Error': rel_error
    }

    df_result = pd.DataFrame(data)

    # Optionally join original metadata
    if original_df is not None:
        original_df = original_df.reset_index(drop=True)
        df_result = pd.concat([original_df.reset_index(drop=True), df_result], axis=1)

    return df_result.head(num_rows)



In [ ]:
# Optionally pass in test_df to enrich the output
prediction_table = get_prediction_table(trained_model, test_dataset, original_df=test_df)



MAE: 2.071796
RMSE: 5.590256
Sum of Relative Errors: 365033.500000


In [ ]:
prediction_table.sort_values(by=['Relative Error'], ascending=False, inplace=True)
prediction_table.head(100)  # Display first 10 rows

,date,UNDERLYING_LAST,DTE,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,moneyness,midprice,T,log_moneyness,q,risk_free_rate,Actual Price,Predicted Price,Absolute Error,Relative Error
392060,2022-12-27,381.38,1.0,368.0,0.01,0.02,9161 x 1058,0.02,0.21602,632.0,1.036359,0.015,0.002740,0.035713,0.015,0.0435,0.015,4.494998,4.479998,298.666504
363901,2022-12-02,406.87,3.0,390.0,0.01,0.02,8772 x 3381,0.02,0.16357,1469.0,1.043256,0.015,0.008219,0.042347,0.015,0.0422,0.015,4.195176,4.180176,278.678406
363906,2022-12-02,406.87,3.0,394.0,0.02,0.03,5766 x 9140,0.03,0.13603,3519.0,1.032665,0.025,0.008219,0.032143,0.015,0.0422,0.025,6.469361,6.444361,257.774445
363900,2022-12-02,406.87,3.0,389.0,0.01,0.02,7849 x 3602,0.02,0.17227,4450.0,1.045938,0.015,0.008219,0.044914,0.015,0.0422,0.015,3.799892,3.784892,252.326141
392059,2022-12-27,381.38,1.0,367.0,0.01,0.02,8596 x 4211,0.01,0.23028,420.0,1.039183,0.015,0.002740,0.038434,0.015,0.0435,0.015,3.714753,3.699753,246.650223
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355561,2022-11-25,402.33,3.0,384.0,0.02,0.03,2839 x 1200,0.03,0.19878,55.0,1.047734,0.025,0.008219,0.046630,0.015,0.0422,0.025,3.153502,3.128501,125.140053
350702,2022-11-23,402.42,2.0,396.0,0.07,0.08,771 x 1336,0.08,0.11287,17807.0,1.016212,0.075,0.005479,0.016082,0.015,0.0421,0.075,9.429302,9.354302,124.724030
396934,2022-12-30,382.44,4.0,363.0,0.02,0.03,3840 x 3687,0.03,0.19629,156.0,1.053554,0.025,0.010959,0.052169,0.015,0.0430,0.025,3.134959,3.109959,124.398361
385231,2022-12-21,386.21,1.0,374.0,0.03,0.04,3415 x 6824,0.04,0.22097,1408.0,1.032647,0.035,0.002740,0.032125,0.015,0.0422,0.035,4.383877,4.348877,124.253639
